In [ ]:
# Install the necessary library if you haven't already:
# !pip install datasets

from datasets import load_dataset
import os

# --- 1. Define File Paths ---
DATA_FILES = {
    'train': 'C:/Users/prash/Documents/AI/Major Project/FineTuning/nepali_XLSum_v2.0/nepali_train.jsonl',
    'test': 'C:/Users/prash/Documents/AI/Major Project/FineTuning/nepali_XLSum_v2.0/nepali_test.jsonl',
    'validation': 'C:/Users/prash/Documents/AI/Major Project/FineTuning/nepali_XLSum_v2.0/nepali_val.jsonl'
}

# --- 2. CONFIGURATION: Edit this variable to choose which file content to display ---
FILE_TO_VIEW = 'validation' # Options: 'train', 'test', or 'validation'

# --- 3. Load All Files ---
try:
    # load_dataset automatically handles JSONL format and loads all splits
    raw_datasets = load_dataset('json', data_files=DATA_FILES)

    # --- 4. Display Counts for All Files ---
    print("--- File Entry Counts ---")
    for split, dataset in raw_datasets.items():
        print(f"File: {split.capitalize():<10} | Entries: {len(dataset):,}")

    # --- 5. Display Content for the Selected File ---
    print(f"\n--- Content Head (Split: {FILE_TO_VIEW.capitalize()}) ---")
    
    if FILE_TO_VIEW in raw_datasets:
        # Display the first 3 entries of the selected split
        print(raw_datasets[FILE_TO_VIEW][:3])
    else:
        print(f"Error: Split '{FILE_TO_VIEW}' not found.")

except Exception as e:
    print(f"\n❌ Error loading dataset: {e}")
    print("Check your file paths and ensure the files are valid JSONL format.")

### Ranking Token Length

In [ ]:
import json
import sentencepiece as spm

# ============================
# CONFIG
# ============================
TOKENIZER_PATH = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\new-tokenizers\bpe-16-updated.model"
INPUT_JSONL = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Filtered\final_train_val_tfidf_filterv2.jsonl"

# ============================
# LOAD TOKENIZER
# ============================
sp = spm.SentencePieceProcessor()
sp.load(TOKENIZER_PATH)
print(f"✓ Tokenizer loaded (vocab size: {sp.vocab_size()})\n")

# ============================
# DATA COLLECTION
# ============================
all_entries = []
total_summary_tokens = 0
total_text_tokens = 0

print(f"Reading and tokenizing file: {INPUT_JSONL}...\n")

with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        if not line.strip():
            continue

        try:
            row = json.loads(line)
            summary_text = row.get("summary", "")
            text = row.get("text", "")

            # Encode and count tokens
            summary_token_count = len(sp.encode(summary_text))
            text_token_count = len(sp.encode(text))

            # Store data
            all_entries.append({
                "line_number": line_number,
                "id": row.get('id', 'N/A'),
                "summary_tokens": summary_token_count,
                "text_tokens": text_token_count,
                "summary": summary_text
            })

            total_summary_tokens += summary_token_count
            total_text_tokens += text_token_count

        except json.JSONDecodeError:
            print(f"Warning: Could not decode JSON at line {line_number}")

# ============================
# SORTING (BY SUMMARY TOKENS)
# ============================
summary_sorted_entries = sorted(all_entries, key=lambda x: x['summary_tokens'])

print("\n" + "=" * 80)
print(f"{'ENTRIES SORTED BY SUMMARY TOKEN LENGTH (ASCENDING)':^80}")
print("=" * 80)
print(f"{'Rank':<5} | {'Line':<6} | {'TextTok':<8} | {'SumTok':<8} | {'ID':<20}")
print("-" * 80)

for rank, entry in enumerate(summary_sorted_entries, start=1):
    print(
        f"{rank:<5} | {entry['line_number']:<6} | "
        f"{entry['text_tokens']:<8} | {entry['summary_tokens']:<8} | "
        f"{str(entry['id'])[:20]:<20}"
    )

# ============================
# SORTING (BY TEXT TOKENS)
# ============================
text_sorted_entries = sorted(all_entries, key=lambda x: x['text_tokens'])

print("\n" + "=" * 80)
print(f"{'ENTRIES SORTED BY TEXT TOKEN LENGTH (ASCENDING)':^80}")
print("=" * 80)
print(f"{'Rank':<5} | {'Line':<6} | {'TextTok':<8} | {'SumTok':<8} | {'ID':<20}")
print("-" * 80)

for rank, entry in enumerate(text_sorted_entries, start=1):
    print(
        f"{rank:<5} | {entry['line_number']:<6} | "
        f"{entry['text_tokens']:<8} | {entry['summary_tokens']:<8} | "
        f"{str(entry['id'])[:20]:<20}"
    )

# ============================
# FINAL STATISTICS
# ============================
if all_entries:
    valid_rows = len(all_entries)
    average_summary_tokens = total_summary_tokens / valid_rows
    average_text_tokens = total_text_tokens / valid_rows

    max_summary_entry = max(all_entries, key=lambda x: x['summary_tokens'])
    min_summary_entry = min(all_entries, key=lambda x: x['summary_tokens'])
    max_text_entry = max(all_entries, key=lambda x: x['text_tokens'])
    min_text_entry = min(all_entries, key=lambda x: x['text_tokens'])

    print("\n" + "=" * 80)
    print(f"{'FINAL SUMMARY STATISTICS':^80}")
    print("=" * 80)
    print(f"Total Entries Processed   : {valid_rows}")
    print(f"Average Summary Tokens    : {average_summary_tokens:.2f}")
    print(f"Average Text Tokens       : {average_text_tokens:.2f}")
    print(f"Maximum Summary Tokens    : {max_summary_entry['summary_tokens']}")
    print(f"Minimum Summary Tokens    : {min_summary_entry['summary_tokens']}")
    print(f"Maximum Text Tokens       : {max_text_entry['text_tokens']}")
    print(f"Minimum Text Tokens       : {min_text_entry['text_tokens']}")
    print("-" * 80)

    print("\n--- SHORTEST SUMMARY ENTRY ---")
    print(f"ID       : {min_summary_entry['id']}")
    print(f"TextTok  : {min_summary_entry['text_tokens']}")
    print(f"SumTok   : {min_summary_entry['summary_tokens']}")
    print(f"Summary  : {min_summary_entry['summary']}")

    print("\n--- LONGEST SUMMARY ENTRY ---")
    print(f"ID       : {max_summary_entry['id']}")
    print(f"TextTok  : {max_summary_entry['text_tokens']}")
    print(f"SumTok   : {max_summary_entry['summary_tokens']}")
    print(f"Summary  : {max_summary_entry['summary'][:200]}...")  # Truncated

    print("\n--- SHORTEST TEXT ENTRY ---")
    print(f"ID       : {min_text_entry['id']}")
    print(f"TextTok  : {min_text_entry['text_tokens']}")
    print(f"SumTok   : {min_text_entry['summary_tokens']}")
    print(f"Text     : {min_text_entry['summary'][:200]}...")  # Optionally truncated

    print("\n--- LONGEST TEXT ENTRY ---")
    print(f"ID       : {max_text_entry['id']}")
    print(f"TextTok  : {max_text_entry['text_tokens']}")
    print(f"SumTok   : {max_text_entry['summary_tokens']}")
    print(f"Text     : {max_text_entry['summary'][:200]}...")  # Optionally truncated

    print("=" * 80)
else:
    print("\nNo valid data found.")

### Side by Side Compare

In [ ]:
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------- CONFIG ----------------
ORIGINAL_JSONL = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Combined\train_plus_val.jsonl"
EDITED_JSONL   = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Edited\tfidf_textrank_sp910.jsonl"

NUM_CONSECUTIVE = 6500   # <-- Set number of entries to compare here
# ----------------------------------------

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def compute_similarity(orig_text, edited_text):
    # --- Cosine similarity using TF-IDF ---
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([orig_text, edited_text])
    cos_sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]

    # --- Jaccard similarity (word-level) ---
    orig_tokens = set(orig_text.split())
    edited_tokens = set(edited_text.split())
    intersection = orig_tokens & edited_tokens
    union = orig_tokens | edited_tokens
    jaccard_sim = len(intersection) / len(union) if union else 0.0

    return cos_sim, jaccard_sim

def main():
    orig = load_jsonl(ORIGINAL_JSONL)
    edited = load_jsonl(EDITED_JSONL)

    num = min(NUM_CONSECUTIVE, len(orig), len(edited))

    for i in range(num):
        o = orig[i]
        e = edited[i]

        # Check that IDs match
        if o["id"] != e["id"]:
            print(f"[ID MISMATCH] Original ID: {o['id']}, Edited ID: {e['id']}")
            continue

        cos_sim, jaccard_sim = compute_similarity(o["text"], e["text"])

        print("=" * 100)
        print(f"[{i+1}] ID: {o['id']}")
        print(f"TITLE: {o.get('title', '')}")
        print("-" * 100)

        print("ORIGINAL TEXT:")
        print(o["text"])
        print("-" * 100)

        print("EXTRACTED TEXT:")
        print(e["text"])
        print("-" * 100)

        print("SUMMARY:")
        print(o["summary"])
        print("-" * 100)

        print(f"Cosine similarity (TF-IDF): {cos_sim:.4f}")
        print(f"Jaccard similarity (word-level): {jaccard_sim:.4f}")
        print("=" * 100)
        print()

    print("Comparison done.")

if __name__ == "__main__":
    main()


### Similarity Calculation of Text and Summary

In [ ]:
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------- CONFIG ----------------
JSONL_FILE = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Combined\train_plus_val.jsonl"
# Number of entries to process (set to None to process all)
NUM_ENTRIES = None
# ----------------------------------------

def load_jsonl(path):
    """Load JSONL file into a list of dicts."""
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def compute_similarity(text, summary):
    """Compute cosine (TF-IDF) and Jaccard similarity between text and summary."""
    # Cosine similarity using TF-IDF
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([text, summary])
    cos_sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]

    # Jaccard similarity (word-level)
    text_tokens = set(text.split())
    summary_tokens = set(summary.split())
    intersection = text_tokens & summary_tokens
    union = text_tokens | summary_tokens
    jaccard_sim = len(intersection) / len(union) if union else 0.0

    return cos_sim, jaccard_sim

def main():
    data = load_jsonl(JSONL_FILE)
    num = NUM_ENTRIES if NUM_ENTRIES is not None else len(data)

    total_cosine = 0.0
    total_jaccard = 0.0

    for i, entry in enumerate(data[:num]):
        text = entry.get("text", "")
        summary = entry.get("summary", "")
        cos_sim, jaccard_sim = compute_similarity(text, summary)

        total_cosine += cos_sim
        total_jaccard += jaccard_sim

        print("=" * 100)
        print(f"[{i+1}] ID: {entry.get('id', '')}")
        print(f"TITLE: {entry.get('title', '')}")
        print("-" * 100)
        print(f"Cosine similarity (TF-IDF): {cos_sim:.4f}")
        print(f"Jaccard similarity (word-level): {jaccard_sim:.4f}")
        print("=" * 100)
        print()

    # Print average similarities
    avg_cosine = total_cosine / num
    avg_jaccard = total_jaccard / num
    print(f"Processed {num} entries.")
    print(f"Average Cosine Similarity: {avg_cosine:.4f}")
    print(f"Average Jaccard Similarity: {avg_jaccard:.4f}")

if __name__ == "__main__":
    main()


### Similarity Comparision

In [ ]:
import json
import sentencepiece as spm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------- CONFIG ----------------
JSONL_FILE1 = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Combined\train_plus_val.jsonl"
JSONL_FILE2 = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Edited\tfidf_textrank_sp910.jsonl"
SP_MODEL_FILE = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\new-tokenizers\bpe-16-updated.model"
NUM_ENTRIES = None  # None to process all entries
# ----------------------------------------

# Load SentencePiece tokenizer
sp = spm.SentencePieceProcessor(model_file=SP_MODEL_FILE)

def load_jsonl(path):
    """Load JSONL file into a list of dicts."""
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def compute_similarity(text, summary):
    """Compute cosine (TF-IDF) and Jaccard similarity between text and summary."""
    # Cosine similarity using TF-IDF
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([text, summary])
    cos_sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]

    # Jaccard similarity (word-level)
    text_tokens = set(text.split())
    summary_tokens = set(summary.split())
    intersection = text_tokens & summary_tokens
    union = text_tokens | summary_tokens
    jaccard_sim = len(intersection) / len(union) if union else 0.0

    return cos_sim, jaccard_sim

def token_length(text):
    """Return number of SentencePiece tokens in the text."""
    return len(sp.encode(text, out_type=int))

def main():
    data1 = load_jsonl(JSONL_FILE1)
    data2 = load_jsonl(JSONL_FILE2)

    num = min(len(data1), len(data2))
    if NUM_ENTRIES is not None:
        num = min(num, NUM_ENTRIES)

    # Initialize totals for averages
    total_cos1 = total_jac1 = 0.0
    total_cos2 = total_jac2 = 0.0

    # Track max/min values
    max_cos1 = max_cos2 = -1; min_cos1 = min_cos2 = 2
    max_jac1 = max_jac2 = -1; min_jac1 = min_jac2 = 2
    max_text_len1 = max_text_len2 = -1; min_text_len1 = min_text_len2 = 1e9
    max_summary_len1 = max_summary_len2 = -1; min_summary_len1 = min_summary_len2 = 1e9

    for i in range(num):
        entry1 = data1[i]
        entry2 = data2[i]

        # Sanity check: IDs match
        if entry1.get("id") != entry2.get("id"):
            print(f"[ID MISMATCH] Entry {i+1}: {entry1.get('id')} vs {entry2.get('id')}")
            continue

        text1, summary1 = entry1.get("text", ""), entry1.get("summary", "")
        text2, summary2 = entry2.get("text", ""), entry2.get("summary", "")

        # Compute similarities
        cos1, jac1 = compute_similarity(text1, summary1)
        cos2, jac2 = compute_similarity(text2, summary2)

        # Token lengths
        len_text1 = token_length(text1)
        len_summary1 = token_length(summary1)
        len_text2 = token_length(text2)
        len_summary2 = token_length(summary2)

        # Update totals
        total_cos1 += cos1
        total_jac1 += jac1
        total_cos2 += cos2
        total_jac2 += jac2

        # Update max/min similarities
        max_cos1 = max(max_cos1, cos1)
        min_cos1 = min(min_cos1, cos1)
        max_cos2 = max(max_cos2, cos2)
        min_cos2 = min(min_cos2, cos2)

        max_jac1 = max(max_jac1, jac1)
        min_jac1 = min(min_jac1, jac1)
        max_jac2 = max(max_jac2, jac2)
        min_jac2 = min(min_jac2, jac2)

        # Update max/min token lengths
        max_text_len1 = max(max_text_len1, len_text1)
        min_text_len1 = min(min_text_len1, len_text1)
        max_summary_len1 = max(max_summary_len1, len_summary1)
        min_summary_len1 = min(min_summary_len1, len_summary1)

        max_text_len2 = max(max_text_len2, len_text2)
        min_text_len2 = min(min_text_len2, len_text2)
        max_summary_len2 = max(max_summary_len2, len_summary2)
        min_summary_len2 = min(min_summary_len2, len_summary2)

        # Print per-entry details
        print("=" * 120)
        print(f"[{i+1}] TITLE: {entry1.get('title', '')}")
        print(f"File1 ID: {entry1.get('id', '')} | Cosine: {cos1:.4f}, Jaccard: {jac1:.4f} | Text tokens: {len_text1}, Summary tokens: {len_summary1}")
        print(f"File2 ID: {entry2.get('id', '')} | Cosine: {cos2:.4f}, Jaccard: {jac2:.4f} | Text tokens: {len_text2}, Summary tokens: {len_summary2}")
        print("=" * 120)
        print()

    # Compute averages
    avg_cos1 = total_cos1 / num
    avg_jac1 = total_jac1 / num
    avg_cos2 = total_cos2 / num
    avg_jac2 = total_jac2 / num

    # Print summary
    print(f"Processed {num} entries.\n")
    print(f"File1 Average Cosine: {avg_cos1:.4f}, Average Jaccard: {avg_jac1:.4f}")
    print(f"File2 Average Cosine: {avg_cos2:.4f}, Average Jaccard: {avg_jac2:.4f}\n")

    print(f"File1 Cosine: max={max_cos1:.4f}, min={min_cos1:.4f}")
    print(f"File2 Cosine: max={max_cos2:.4f}, min={min_cos2:.4f}")
    print(f"File1 Jaccard: max={max_jac1:.4f}, min={min_jac1:.4f}")
    print(f"File2 Jaccard: max={max_jac2:.4f}, min={min_jac2:.4f}\n")

    print(f"File1 Text tokens: max={max_text_len1}, min={min_text_len1}")
    print(f"File2 Text tokens: max={max_text_len2}, min={min_text_len2}")
    print(f"File1 Summary tokens: max={max_summary_len1}, min={min_summary_len1}")
    print(f"File2 Summary tokens: max={max_summary_len2}, min={min_summary_len2}")

if __name__ == "__main__":
    main()
